# Transaction Node Formulation with Size Buckets

This notebook follows the same graph layout as `Preprocess-ProjNodeFormulation.ipynb`, but node IDs are assigned to `(project_id, size_bucket)` pairs instead of only `project_id`.

Example: if project 0 has `SZ2` and `SZ4`, and project 1 has `SZ1`, `SZ2`, `SZ5`, then the node IDs are assigned consecutively as 0, 1, 2, 3, 4.

In [8]:
import os
from pathlib import Path

import pandas as pd
from tqdm import tqdm

DB_FOLDER = r"dataset\database_v3"
GRAPH_FOLDER = os.path.join(DB_FOLDER, "Graph_Size")
TIMESTEP_FOLDER = os.path.join(GRAPH_FOLDER, "timestep")
NODE_FOLDER = os.path.join(GRAPH_FOLDER, "nodes")

os.makedirs(GRAPH_FOLDER, exist_ok=True)
os.makedirs(TIMESTEP_FOLDER, exist_ok=True)
os.makedirs(NODE_FOLDER, exist_ok=True)

TRANSACTION_PATH = os.path.join(DB_FOLDER, "transaction.csv")
N_SIZE_BUCKETS = 5

In [9]:
def make_size_bucket(area_sqft, q=5):
    """Create global equal-frequency sqft buckets.

    SZ1 is the smallest sqft group and SZq is the largest sqft group. With q=5,
    the function tries to place roughly 20% of valid transaction records in each
    bucket. Missing sqft values are assigned to SZ1 so every row has a node.
    """
    s_valid = area_sqft.dropna()
    out = pd.Series(index=area_sqft.index, dtype="object")

    if s_valid.nunique() < 2:
        out[:] = "SZ1"
        return out

    q_use = min(q, int(s_valid.nunique()))
    labels = [f"SZ{i+1}" for i in range(q_use)]

    try:
        buckets = pd.qcut(
            s_valid.rank(method="first"),
            q=q_use,
            labels=labels,
            duplicates="drop",
        )
    except Exception:
        buckets = pd.cut(
            s_valid,
            bins=q_use,
            labels=labels,
            include_lowest=True,
            duplicates="drop",
        )

    out.loc[s_valid.index] = buckets.astype(str)
    return out.fillna("SZ1")


def explain_size_buckets(df, sqft_col="sqft", bucket_col="size_bucket"):
    summary = (
        df.groupby(bucket_col, dropna=False)[sqft_col]
        .agg(
            n_records="size",
            missing_sqft=lambda s: s.isna().sum(),
            min_sqft="min",
            median_sqft="median",
            max_sqft="max",
        )
        .sort_index()
    )
    display(summary)
    return summary

In [10]:
trans_df = pd.read_csv(TRANSACTION_PATH)

trans_df["sqft"] = pd.to_numeric(trans_df["sqft"], errors="coerce")
trans_df["rent_per_sqft"] = pd.to_numeric(trans_df["rent_per_sqft"], errors="coerce")
trans_df["No of Bedroom"] = pd.to_numeric(trans_df["No of Bedroom"], errors="coerce")
trans_df["Lease Commencement Date"] = pd.to_datetime(
    trans_df["Lease Commencement Date"], errors="coerce"
)

print(f"Loaded {len(trans_df):,} transaction records")
display(trans_df.head())

Loaded 1,260,795 transaction records


,transaction_id,timestep,Lease Commencement Date,project_id,Project Name,No of Bedroom,Floor Area (SQFT),sqft,Monthly Rent ($),rent_per_sqft
0,1,0,1999-11-01,1139,MANDARIN GARDENS,NaN,700 to 800,750.0,2015,2.686667
1,2,0,1999-11-01,1139,MANDARIN GARDENS,NaN,700 to 800,750.0,2325,3.100000
2,3,0,1999-11-01,1528,REGENT PARK,NaN,"1,100 to 1,200",1150.0,2500,2.173913
3,4,0,1999-11-01,276,CANNE LODGE,NaN,800 to 900,850.0,1700,2.000000
4,5,0,1999-11-01,276,CANNE LODGE,NaN,800 to 900,850.0,1500,1.764706


In [11]:
# Fill missing bedrooms using the most common bedroom count for the same project and sqft.
trans_df["No of Bedroom"] = trans_df["No of Bedroom"].fillna(0)

majority_map = (
    trans_df.loc[trans_df["No of Bedroom"] != 0]
    .groupby(["project_id", "sqft"])["No of Bedroom"]
    .agg(lambda x: x.mode().iloc[0])
)

missing_bedroom_mask = trans_df["No of Bedroom"] == 0
trans_df.loc[missing_bedroom_mask, "No of Bedroom"] = trans_df.loc[
    missing_bedroom_mask, ["project_id", "sqft"]
].apply(lambda row: majority_map.get((row["project_id"], row["sqft"]), 0), axis=1)

print(f"Remaining bedroom=0 rows: {(trans_df['No of Bedroom'] == 0).sum():,}")

Remaining bedroom=0 rows: 77,847


In [12]:
# Global equal-frequency size buckets, matching the Vinh notebook method.
trans_df["size_bucket"] = make_size_bucket(trans_df["sqft"], q=N_SIZE_BUCKETS)
size_bucket_summary = explain_size_buckets(trans_df)

trans_df["node_key"] = trans_df["project_id"].astype(str) + " | " + trans_df["size_bucket"]
display(trans_df[["project_id", "Project Name", "sqft", "size_bucket", "node_key"]].head())

,n_records,missing_sqft,min_sqft,median_sqft,max_sqft
size_bucket,,,,,
SZ1,252159,0,350.0,550.0,650.0
SZ2,252159,0,650.0,850.0,950.0
SZ3,252159,0,950.0,1050.0,1150.0
SZ4,252159,0,1150.0,1250.0,1450.0
SZ5,252159,0,1450.0,1750.0,3250.0


,project_id,Project Name,sqft,size_bucket,node_key
0,1139,MANDARIN GARDENS,750.0,SZ2,1139 | SZ2
1,1139,MANDARIN GARDENS,750.0,SZ2,1139 | SZ2
2,1528,REGENT PARK,1150.0,SZ3,1528 | SZ3
3,276,CANNE LODGE,850.0,SZ2,276 | SZ2
4,276,CANNE LODGE,850.0,SZ2,276 | SZ2


In [13]:
# node_id is assigned to each observed (project_id, size_bucket) pair.
node_id_df = (
    trans_df.groupby(["project_id", "Project Name", "size_bucket", "node_key"], dropna=False)
    .agg(count=("transaction_id", "size"))
    .reset_index()
    .sort_values(["project_id", "size_bucket"])
    .reset_index(drop=True)
)

node_id_df.insert(0, "node_id", node_id_df.index)
node_id_df = node_id_df[["node_id", "project_id", "Project Name", "size_bucket", "node_key", "count"]]
node_id_df.to_csv(os.path.join(GRAPH_FOLDER, "node_id.csv"), index=False)

print(f"Saved node_id.csv with {len(node_id_df):,} project-size nodes")
display(node_id_df.head(10))

Saved node_id.csv with 7,958 project-size nodes


,node_id,project_id,Project Name,size_bucket,node_key,count
0,0,0,# 1 LOFT,SZ1,0 | SZ1,229
1,1,0,# 1 LOFT,SZ2,0 | SZ2,13
2,2,0,# 1 LOFT,SZ3,0 | SZ3,12
3,3,0,# 1 LOFT,SZ4,0 | SZ4,16
4,4,1,# 1 SUITES,SZ1,1 | SZ1,240
5,5,1,# 1 SUITES,SZ2,1 | SZ2,13
6,6,2,1 KING ALBERT PARK,SZ2,2 | SZ2,86
7,7,2,1 KING ALBERT PARK,SZ3,2 | SZ3,100
8,8,2,1 KING ALBERT PARK,SZ4,2 | SZ4,133
9,9,2,1 KING ALBERT PARK,SZ5,2 | SZ5,53


In [14]:
# Aggregate transaction records into node-timestep observations.
nodes_df = (
    trans_df.groupby(
        ["node_key", "project_id", "Project Name", "size_bucket", "timestep"],
        dropna=False,
    )
    .agg(
        lease_commencement_date=("Lease Commencement Date", "first"),
        no_of_bedroom=("No of Bedroom", "mean"),
        sqft=("sqft", "mean"),
        rent_per_sqft=("rent_per_sqft", "mean"),
        n_transactions=("transaction_id", "size"),
    )
    .reset_index()
)

nodes_df = nodes_df.merge(
    node_id_df[["node_id", "node_key"]],
    on="node_key",
    how="left",
)

nodes_df = nodes_df[
    [
        "node_id",
        "node_key",
        "project_id",
        "Project Name",
        "size_bucket",
        "timestep",
        "lease_commencement_date",
        "no_of_bedroom",
        "sqft",
        "rent_per_sqft",
        "n_transactions",
    ]
].sort_values(["node_id", "timestep"]).reset_index(drop=True)

nodes_df.to_csv(os.path.join(GRAPH_FOLDER, "nodes.csv"), index=False)
print(f"Saved nodes.csv with {len(nodes_df):,} observed node-timestep rows")
display(nodes_df.head())

Saved nodes.csv with 497,885 observed node-timestep rows


,node_id,node_key,project_id,Project Name,size_bucket,timestep,lease_commencement_date,no_of_bedroom,sqft,rent_per_sqft,n_transactions
0,0,0 | SZ1,0,# 1 LOFT,SZ1,197,2016-04-01,1.0,450.000000,4.123457,9
1,0,0 | SZ1,0,# 1 LOFT,SZ1,198,2016-05-01,1.0,461.111111,4.179574,9
2,0,0 | SZ1,0,# 1 LOFT,SZ1,199,2016-06-01,1.0,450.000000,4.361111,4
3,0,0 | SZ1,0,# 1 LOFT,SZ1,200,2016-07-01,1.0,450.000000,4.333333,2
4,0,0 | SZ1,0,# 1 LOFT,SZ1,201,2016-08-01,1.0,450.000000,4.111111,2


In [15]:
# Create one CSV per timestep, with missing timesteps interpolated within each node.
df_use = nodes_df.copy()
df_use["y_mask"] = 1

filled_list = []

for node_id, group in df_use.groupby("node_id"):
    group = group.sort_values("timestep").copy()

    node_key = group["node_key"].iloc[0]
    project_id = group["project_id"].iloc[0]
    project_name = group["Project Name"].iloc[0]
    size_bucket = group["size_bucket"].iloc[0]

    full_timestep = pd.DataFrame(
        {"timestep": range(int(group["timestep"].min()), int(group["timestep"].max()) + 1)}
    )

    node_df = full_timestep.merge(group, on="timestep", how="left")

    node_df["node_id"] = node_id
    node_df["node_key"] = node_df["node_key"].fillna(node_key)
    node_df["project_id"] = node_df["project_id"].fillna(project_id)
    node_df["Project Name"] = node_df["Project Name"].fillna(project_name)
    node_df["size_bucket"] = node_df["size_bucket"].fillna(size_bucket)
    node_df["y_mask"] = node_df["y_mask"].fillna(0).astype(int)
    node_df["n_transactions"] = node_df["n_transactions"].fillna(0).astype(int)

    node_df["no_of_bedroom"] = node_df["no_of_bedroom"].interpolate(method="linear")
    node_df["sqft"] = node_df["sqft"].interpolate(method="linear")
    node_df["rent_per_sqft"] = node_df["rent_per_sqft"].interpolate(method="linear")

    filled_list.append(node_df)

filled_df = pd.concat(filled_list, ignore_index=True)

timestep_date_map = (
    nodes_df[["timestep", "lease_commencement_date"]]
    .drop_duplicates(subset=["timestep"])
    .sort_values("timestep")
)

filled_df = filled_df.drop(columns=["lease_commencement_date"], errors="ignore")
filled_df = filled_df.merge(timestep_date_map, on="timestep", how="left")
filled_df = filled_df.sort_values(["timestep", "node_id"]).reset_index(drop=True)

for t, group in filled_df.groupby("timestep"):
    group = group.sort_values("node_id")
    date = group["lease_commencement_date"].iloc[0]
    date_str = date.strftime("%Y-%m") if pd.notna(date) else "unknown"
    file_path = os.path.join(TIMESTEP_FOLDER, f"{int(t)}_{date_str}.csv")

    group_to_save = group[
        [
            "timestep",
            "node_id",
            "node_key",
            "project_id",
            "Project Name",
            "size_bucket",
            "no_of_bedroom",
            "sqft",
            "n_transactions",
            "rent_per_sqft",
            "y_mask",
        ]
    ]
    group_to_save.to_csv(file_path, index=False)

print(f"Saved {filled_df['timestep'].nunique():,} timestep files to {TIMESTEP_FOLDER}")

Saved 315 timestep files to dataset\database_v3\Graph_Size\timestep


In [16]:
# Merge project-level and macro-level features into each timestep file, matching Graph_Proj.
project_df = pd.read_csv(os.path.join(DB_FOLDER, "project_processed.csv"))
school_df = pd.read_csv(os.path.join(DB_FOLDER, "school_processed.csv"))
mrt_df = pd.read_csv(os.path.join(DB_FOLDER, "mrt_processed.csv"))
proximity_df = pd.read_csv(os.path.join(DB_FOLDER, "proximity_processed.csv"))
amenities_df = pd.read_csv(os.path.join(DB_FOLDER, "amenities.csv"))
macro_df = pd.read_csv(os.path.join(DB_FOLDER, "macro_data_v1_processed.csv"))

project_df = project_df.drop_duplicates(subset=["project_id"])
school_df = school_df.drop_duplicates(subset=["project_id"])
mrt_df = mrt_df.drop_duplicates(subset=["project_id"])
proximity_df = proximity_df.drop_duplicates(subset=["project_id"])
amenities_df = amenities_df.drop_duplicates(subset=["project_id"])
macro_df = macro_df.drop_duplicates(subset=["timestep"])

project_feature_dfs = [project_df, school_df, mrt_df, proximity_df, amenities_df]

csv_files = sorted(Path(TIMESTEP_FOLDER).glob("*.csv"))
for csv_path in tqdm(csv_files, desc="Merging timestep files"):
    df = pd.read_csv(csv_path)

    for feature_df in project_feature_dfs:
        df = df.merge(feature_df, on="project_id", how="left", suffixes=("", "_dup"))
        dup_cols = [col for col in df.columns if col.endswith("_dup")]
        df = df.drop(columns=dup_cols)

    df = df.merge(macro_df, on="timestep", how="left", suffixes=("", "_dup"))
    dup_cols = [col for col in df.columns if col.endswith("_dup")]
    df = df.drop(columns=dup_cols)

    last_cols = ["rent_per_sqft", "y_mask"]
    other_cols = [col for col in df.columns if col not in last_cols]
    df = df[other_cols + last_cols]
    df.to_csv(csv_path, index=False)

print("Finished merging features into all timestep CSV files.")

Merging timestep files: 100%|██████████| 315/315 [02:21<00:00,  2.22it/s]

Finished merging features into all timestep CSV files.


In [17]:
# Create one CSV per node from the timestep files, including rows where y_mask = 0.
csv_files = sorted(Path(TIMESTEP_FOLDER).glob("*.csv"))
all_dfs = []

for csv_path in tqdm(csv_files, desc="Reading timestep files"):
    all_dfs.append(pd.read_csv(csv_path))

all_df = pd.concat(all_dfs, ignore_index=True)
all_df = all_df.sort_values(["node_id", "timestep"])

for node_id, node_df in tqdm(all_df.groupby("node_id"), desc="Saving node files"):
    node_df = node_df.sort_values("timestep").reset_index(drop=True)
    file_path = Path(NODE_FOLDER) / f"{int(node_id):04d}.csv"
    node_df.to_csv(file_path, index=False)

print(f"Finished saving node files to: {NODE_FOLDER}")
print(f"Number of node files: {all_df['node_id'].nunique():,}")

Saving node files: 100%|██████████| 7958/7958 [03:12<00:00, 41.28it/s]

Finished saving node files to: dataset\database_v3\Graph_Size\nodes
Number of node files: 7,958


## CCR Node ID Filtering

In [18]:
ccr_districts = [1, 2, 9, 10, 11]

node_path = os.path.join(GRAPH_FOLDER, "node_id.csv")
project_path = os.path.join(DB_FOLDER, "project.csv")
output_path = os.path.join(GRAPH_FOLDER, "node_id_ccr.csv")

node_df = pd.read_csv(node_path)
project_df = pd.read_csv(project_path)
project_map = project_df[["project_id", "Postal District"]].drop_duplicates("project_id")

node_ccr_df = node_df.merge(project_map, on="project_id", how="left")
node_ccr_df = node_ccr_df[node_ccr_df["Postal District"].isin(ccr_districts)].copy()
node_ccr_df = node_ccr_df[node_df.columns]
node_ccr_df.to_csv(output_path, index=False)

print(f"Saved CCR node file to: {output_path}")
print(f"Number of CCR nodes: {len(node_ccr_df):,}")

Saved CCR node file to: dataset\database_v3\Graph_Size\node_id_ccr.csv
Number of CCR nodes: 2,314
